# SnowEx Lambda Demo

**CloudBank Cloud Clinic — Live Demo**

This notebook demonstrates querying the SnowEx PostgreSQL database through AWS Lambda using a simple Python client.

---

## 1. Setup: Import Libraries

We need two things:
- `SnowExLambdaClient` — connects to our Lambda function
- `matplotlib` — for visualization

**Key point:** Notice there's no configuration step. No credentials file, no AWS keys, no connection strings.

In [ ]:
from snowexsql.api import SnowExLambdaClient
import matplotlib.pyplot as plt

# Set figure size for better visibility
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 2. Connect to the Database

Instantiate the client. Behind the scenes, this:
- Uses your AWS credentials (via IAM)
- Sends requests to the Lambda Function URL
- Lambda retrieves database credentials from Secrets Manager
- Lambda queries PostgreSQL on EC2

From your perspective: **one line of code.**

In [ ]:
client = SnowExLambdaClient()

## 3. Query Snow Depth Data

Let's retrieve snow depth measurements from **Grand Mesa, Colorado** during the **February 2020** field campaign.

**Query parameters:**
- `campaign`: "Grand Mesa 2020" — one of SnowEx's core study sites
- `instrument`: "magnaprobe" — handheld GPS-enabled snow depth probe
- `limit`: 500 — keep the query fast for demo purposes

**Expected behavior:**
- First invocation may take 3-5 seconds (Lambda cold start)
- Returns a pandas DataFrame with measurements

In [ ]:
data = client.get_point_measurements(
    campaign="Grand Mesa 2020",
    instrument="magnaprobe",
    limit=500
)

## 4. Inspect the Results

The Lambda function returns data as a pandas DataFrame — standard scientific Python workflow.

In [ ]:
print(f"Retrieved {len(data)} measurements\n")
print("First few rows:")
data[['date', 'depth', 'latitude', 'longitude']].head()

## 5. Visualize: Snow Depth Map

Create a simple scatter plot showing:
- **X-axis:** Longitude
- **Y-axis:** Latitude
- **Color:** Snow depth (cm) — light blue = shallow, dark blue = deep

This gives us a spatial view of snow distribution across the field site.

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    data.longitude, 
    data.latitude, 
    c=data.depth, 
    cmap='Blues', 
    s=50,
    alpha=0.7,
    edgecolors='black',
    linewidth=0.5
)
plt.colorbar(scatter, label='Snow Depth (cm)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Grand Mesa Snow Depth — February 2020', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Summary

**What we just demonstrated:**

✅ **No infrastructure setup** — no servers, no SSH, no VPN  
✅ **No credential management** — no passwords, no config files  
✅ **Standard workflow** — pandas DataFrame, matplotlib plotting  
✅ **Real data** — 500 field measurements from PostgreSQL database  

**The architecture behind this:**
1. Client sends query → Function URL (HTTPS)
2. AWS IAM authenticates the request
3. Lambda invokes container image from ECR
4. Lambda retrieves DB credentials from Secrets Manager
5. Lambda queries PostgreSQL on EC2
6. Results return as JSON → converted to pandas DataFrame

**Transferable to:**
- Genomics databases
- Climate model outputs
- Hydrology data
- Any PostgreSQL dataset that needs secure API access

---

**Resources:**
- SnowEx SQL GitHub: [github.com/SnowEx/snowexsql](https://github.com/SnowEx/snowexsql)
- Project Pythia Tutorial: [projectpythia.org/snow-observations-cookbook](https://projectpythia.org/snow-observations-cookbook/notebooks/snowexsql-database/)
- AWS Lambda Docs: [docs.aws.amazon.com/lambda](https://docs.aws.amazon.com/lambda/)